# Power and pilot studies

Companion to the course pages *Sample size and power* and *How to: plan a sample size*.

Everything here is simulated from a known true effect, so you can see:

1. How **power** grows with sample size (and that a simulation agrees with the textbook formula).
2. Why the effect size from a **small pilot** is a poor input for a power calculation: the pilots that look convincing are the ones that exaggerate.
3. What happens to a study **planned from a pilot's effect size**.

Change the parameters in the next cell and use *Run All*. Needs `numpy`, `scipy`, `statsmodels`, `matplotlib`.

In [ ]:
# ---- Parameters: change these and re-run ----
TRUE_D = 0.3          # true standardized effect (Cohen's d = difference / SD); 0.3 is a modest effect
PILOT_N = 10          # units per group in each pilot study
ALPHA = 0.05          # significance level
TARGET_POWER = 0.8    # power you aim for when planning the full study
N_GRID = [10, 20, 40, 80, 120, 160, 240]   # per-group sample sizes for experiment 1
N_SIMS = 4000         # simulated studies per setting
SEED = 0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import TTestIndPower

rng = np.random.default_rng(SEED)
power_calc = TTestIndPower()


def simulate_studies(n, d, n_sims=N_SIMS):
    """Simulate n_sims two-group studies (n per group); return observed d and p-value for each."""
    control = rng.normal(0, 1, (n_sims, n))
    treated = rng.normal(d, 1, (n_sims, n))
    p = stats.ttest_ind(treated, control, axis=1).pvalue
    pooled_sd = np.sqrt((treated.var(axis=1, ddof=1) + control.var(axis=1, ddof=1)) / 2)
    d_hat = (treated.mean(axis=1) - control.mean(axis=1)) / pooled_sd
    return d_hat, p

## 1. Power curve: simulation vs. formula

Power is the share of simulated studies that reach p < alpha. The dots are simulated; the line is the closed-form answer from `statsmodels`. They should agree, which is why simulation is a trustworthy fallback for designs that have no formula.

In [ ]:
simulated = [np.mean(simulate_studies(n, TRUE_D)[1] < ALPHA) for n in N_GRID]
formula = power_calc.power(effect_size=TRUE_D, nobs1=np.array(N_GRID), alpha=ALPHA)
needed = power_calc.solve_power(effect_size=TRUE_D, alpha=ALPHA, power=TARGET_POWER)
print(f"Per-group n for {TARGET_POWER:.0%} power at d = {TRUE_D}: {int(np.ceil(needed))}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(N_GRID, formula, label="formula")
ax.plot(N_GRID, simulated, "o", label="simulation")
ax.axhline(TARGET_POWER, color="gray", linestyle=":")
ax.set_xlabel("units per group")
ax.set_ylabel("power")
ax.legend()
plt.show()

## 2. What a small pilot tells you about the effect

We run many pilots of `PILOT_N` units per group, all from the same true effect. Look at (a) how widely the observed effect size varies, and (b) what happens if you only look at the pilots that came out statistically significant, which are the ones most likely to get written up and followed up.

In [ ]:
d_hat, p = simulate_studies(PILOT_N, TRUE_D)
significant = p < ALPHA

print(f"True d: {TRUE_D}")
print(f"Pilots reaching p < {ALPHA} (= power at n = {PILOT_N}): {significant.mean():.0%}")
print(f"Observed d, all pilots:          mean {d_hat.mean():.2f}, middle 90% from {np.percentile(d_hat, 5):.2f} to {np.percentile(d_hat, 95):.2f}")
print(f"Observed d, significant pilots:  mean {np.abs(d_hat[significant]).mean():.2f}  "
      f"(exaggeration factor {np.abs(d_hat[significant]).mean() / TRUE_D:.1f}x)")
print(f"Significant pilots with the WRONG sign: {(d_hat[significant] < 0).mean():.1%}")

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(d_hat.min(), d_hat.max(), 50)
ax.hist(d_hat[~significant], bins=bins, alpha=0.6, label="not significant")
ax.hist(d_hat[significant], bins=bins, alpha=0.6, label="significant")
ax.axvline(TRUE_D, color="green", linestyle="--", label="true d")
ax.set_xlabel("observed d in the pilot")
ax.set_ylabel("number of pilots")
ax.legend()
plt.show()

## 3. Planning the full study from the pilot's effect size

Suppose each pilot's observed effect is taken at face value to size the full study (rough per-group n from the normal approximation). We then ask what power that full study really has, given the *true* effect. We repeat this for all pilots and for only the significant ones.

In [ ]:
# Rough per-group n from the pilot's d (normal approximation), capped at 1000 to keep the numbers sensible.
z = stats.norm.ppf(1 - ALPHA / 2) + stats.norm.ppf(TARGET_POWER)
planned_n = np.clip(np.ceil(2 * (z / np.maximum(np.abs(d_hat), 1e-6)) ** 2), 2, 1000)
achieved_power = power_calc.power(effect_size=TRUE_D, nobs1=planned_n, alpha=ALPHA)

print(f"Per-group n that gives {TARGET_POWER:.0%} power at the TRUE d: {int(np.ceil(needed))}")
for label, mask in [("all pilots", np.ones_like(significant)), ("only significant pilots", significant)]:
    print(f"\n{label}:")
    print(f"  planned n per group: median {np.median(planned_n[mask]):.0f}, middle 90% from "
          f"{np.percentile(planned_n[mask], 5):.0f} to {np.percentile(planned_n[mask], 95):.0f}")
    print(f"  true power of the planned study: median {np.median(achieved_power[mask]):.0%}")
    print(f"  planned studies with true power below {TARGET_POWER:.0%}: {(achieved_power[mask] < TARGET_POWER).mean():.0%}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(achieved_power[significant], bins=20, alpha=0.6, label="planned from significant pilots")
ax.hist(achieved_power, bins=20, alpha=0.6, label="planned from all pilots")
ax.axvline(TARGET_POWER, color="gray", linestyle=":")
ax.set_xlabel("true power of the planned study")
ax.set_ylabel("number of pilots")
ax.legend()
plt.show()

## Things to try

- Increase `PILOT_N` to 100: the exaggeration among significant pilots falls from about 4x to about 1.4x and the planned sample sizes become much less erratic. The problem is specific to small, noisy pilots.
- Set `TRUE_D = 0.8` (a large effect): a pilot of 10 per group detects it about 40% of the time, and the exaggeration is smaller (about 1.7x). Inflation matters most for modest effects.
- Set `TRUE_D = 0.1`: only about 6% of pilots are significant, and about a quarter of those have the wrong sign.
- The alternative to trusting a pilot: pick the smallest effect that would matter in practice and plan for that (the how-to page's sensitivity table). Try `TRUE_D = 0.3` but plan with `d = 0.3` directly, and compare to the planned sample sizes from the pilots above.

Note that a pilot is still valuable for checking logistics and estimating the SD and the dropout rate. It is the *effect size* from a small pilot that shouldn't be taken at face value.